Imports

In [1]:

import numpy as np

from Background_functions import read_audio_file, split_audio_segments, start_end_times
from STFT import resample_audio, stft_calculation, plot_spectrogram
from DFT import dp

from pathlib import Path
from scipy.spatial import distance as dist

DATA_22 = Path(r"C:\Users\HP\Desktop\Skripsie data\DataSubmission\2023_04_22")

Data download

In [2]:
audio = DATA_22 / "20230422_171301.WAV"
f_s, x = read_audio_file(audio)
duration = len(x) / f_s  # Duration of the audio in seconds

start = 0.0  # Start time in seconds
window = 3.0  # Window length in seconds
start_t = []
end_t = []

while start < duration:
    end = min(start + window, duration)  # Ensure we don't exceed the audio duration
    start_t.append(start)
    end_t.append(end)
    start += window  # Move to the next window
length = len(start_t)
print(length)

audio_segments = split_audio_segments(x, f_s, start_t, end_t)

89479


Template

In [3]:
text = DATA_22 / "20230422_171301.Detections.selections.txt"
temp_marks, start_t_temp, end_t_temp = start_end_times(text)

whale_call_segments = split_audio_segments(x, f_s, start_t_temp, end_t_temp)

audio_template_1 = whale_call_segments[0]

STFT Model

In [4]:
def short_time_calc(temp, sig, fs):
    fs_new = 2000
    resampled_sig = resample_audio(sig, fs, fs_new)
    resampled_temp = resample_audio(temp, fs, fs_new)
    f_seg, t_seg, Zxx_seg = stft_calculation(resampled_sig, fs, fs_new)
    f_temp, t_temp, Zxx_temp = stft_calculation(resampled_temp, fs, fs_new)
    #plot_spectrogram(f_seg, t_seg, Zxx_seg, fs_new)
    return Zxx_seg, Zxx_temp

DTW Model

In [5]:
def DTW_calc(template_Zxx, comparison_Zxx):
    x_seq = np.abs(template_Zxx).T      # shape: (n_time_frames, n_freq_bins)
    y_seq = np.abs(comparison_Zxx).T    # shape: (n_time_frames, n_freq_bins)

    dist_mat = dist.cdist(x_seq, y_seq, "cosine")
    path, cost_mat = dp(dist_mat)
    ali_cost = cost_mat[-1, -1]
    #print("Alignment cost: {:.4f}".format(ali_cost))

    M = x_seq.shape[0]
    N = y_seq.shape[0]
    norm_ali_cost = ali_cost / (M + N)
    #print("Normalized alignment cost: {:.4f}".format(norm_ali_cost))
    return norm_ali_cost

Running loop for comparison

In [6]:
stft_cost = []
for i in range(100):
    audio_seg = audio_segments[i]
    Zxx_seg, Zxx_temp = short_time_calc(audio_template_1, audio_seg, f_s)
    stft_cost.append(DTW_calc(Zxx_temp, Zxx_seg))
    print(i)


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
